# Benchmark de encoders - CODEFEST Ad Astra 2026

Compara tres modelos *encoder-only* multilingües para decidir cuál usar en la Fase 4.

| Modelo | dim | arquitectura | prefijos |
|---|---|---|---|
| `ibm-granite/granite-embedding-311m-multilingual-r2` | 768 | ModernBERT | no |
| `ibm-granite/granite-embedding-97m-multilingual-r2` | 384 | ModernBERT | no |
| `intfloat/multilingual-e5-base` | 768 | XLM-RoBERTa | sí (`query:` / `passage:`) |

Los tres se verificaron como *encoder-only* en su `config.json`, como exige la Sección 8.3 del reto.

**Cuatro bloques:**

1. **Ficha técnica** — qué dice cada modelo de sí mismo (dimensión, límite de tokens real, tamaño).
2. **Idiomas cruzados** — si una consulta en español encuentra un documento en inglés.
3. **Prefijos de e5** — cuánto se pierde si se olvidan.
4. **Tiempo en CPU** — cuánto tarda indexar el corpus completo.

**Antes de correr:** todo esto es CPU. El bloque 1 descarga los tres modelos (~2.5 GB en total) y eso puede tomar varios minutos la primera vez. Los bloques siguientes ya usan la copia local.

## 0. Preparación

### Instalar dependencias

In [ ]:
# !pip install sentence-transformers torch pandas

### Cómo se descargan los modelos

En HuggingFace no hay un botón de "descargar" porque no se baja un archivo suelto: un modelo es una carpeta con los pesos, el tokenizer y varios archivos de configuración. La librería los baja sola la primera vez que nombras el modelo, y los guarda en una caché local:

```
~/.cache/huggingface/hub/          (Linux y macOS)
C:\Users\<usuario>\.cache\huggingface\hub\   (Windows)
```

De ahí en adelante los reutiliza sin volver a bajar nada, aunque cierres el notebook.

La celda siguiente hace la descarga de forma explícita, para que veas el progreso y sepas cuánto ocupa cada uno. `ignore_patterns` evita bajar las versiones ONNX y OpenVINO que traen algunos repos: son copias del mismo modelo en otro formato y duplicarían el peso sin que las uses.

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download

MODELOS = {
    "granite-311m": {
        "repo": "ibm-granite/granite-embedding-311m-multilingual-r2",
        "prefijo_consulta": "",
        "prefijo_texto": "",
    },
    "granite-97m": {
        "repo": "ibm-granite/granite-embedding-97m-multilingual-r2",
        "prefijo_consulta": "",
        "prefijo_texto": "",
    },
    "e5-base": {
        "repo": "intfloat/multilingual-e5-base",
        "prefijo_consulta": "query: ",
        "prefijo_texto": "passage: ",
    },
}

SOBRAN = ["onnx/*", "openvino/*", "*.onnx", "*.h5", "*.msgpack", "*.ot", "*.tflite"]

# Los modelos granite son ModernBERT, soportado desde transformers 4.48.
# Con una versión anterior fallan al cargar con un error poco claro.
import transformers
print("transformers", transformers.__version__, "(los granite necesitan >= 4.48)\n")

def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6

for alias, cfg in MODELOS.items():
    print(f"--- {alias} ---")
    ruta = snapshot_download(cfg["repo"], ignore_patterns=SOBRAN)
    cfg["ruta_local"] = ruta
    print(f"    {tamano_mb(ruta):,.0f} MB en {ruta}\n")

### Cargar el mini-corpus

Doce fragmentos reales del corpus y tres consultas reales del reto (q006, q026, q042), con relevancia graduada de 0 a 3 asignada a mano. Un fragmento que no aparece en el campo `relevance` de una consulta tiene relevancia 0 para esa consulta.

Fíjate en algo que va a importar en el bloque 2: **las tres consultas están en español, pero los fragmentos correctos de q006 y q026 están en inglés.** Eso no es un descuido del fixture, es exactamente el escenario del reto.

In [ ]:
import json
from pathlib import Path

def raiz_repo(inicio=None):
    '''Sube por el árbol de carpetas hasta encontrar la raíz del repo.'''
    actual = Path(inicio or Path.cwd()).resolve()
    for candidata in [actual, *actual.parents]:
        if (candidata / "tests" / "fixtures" / "mini_corpus.json").exists():
            return candidata
    raise FileNotFoundError(
        "No encuentro tests/fixtures/mini_corpus.json. "
        "Abre el notebook desde dentro del repo, o fija RAIZ a mano."
    )

RAIZ = raiz_repo()
CORPUS = json.loads((RAIZ / "tests" / "fixtures" / "mini_corpus.json").read_text(encoding="utf-8"))

CONSULTAS = CORPUS["consultas"]
TEXTOS = CORPUS["texts"]

print(f"raíz del repo: {RAIZ}")
print(f"{len(CONSULTAS)} consultas, {len(TEXTOS)} fragmentos")
print()
for c in CONSULTAS:
    oro = [t["id"] for t in TEXTOS if t["relevance"].get(c["id"], 0) >= 2]
    idiomas = {t["language"] for t in TEXTOS if t["id"] in oro}
    print(f"{c['id']}  correctos: {len(oro)} ({', '.join(sorted(idiomas))})  |  {c['text'][:70]}...")
print()
print("fragmentos por idioma:", {i: sum(1 for t in TEXTOS if t["language"] == i) for i in {t["language"] for t in TEXTOS}})

## Bloque 1 - Ficha técnica

Sin inferencia todavía: solo abrir cada modelo y preguntarle qué es. Interesan cuatro cosas.

**`max_seq_length`** es el dato que de verdad manda. Un modelo puede tener una arquitectura que soporte 32.000 tokens y aun así venir configurado para cortar en 512, porque el archivo `sentence_bert_config.json` lo fija ahí. Si le pasas un texto más largo, **lo trunca en silencio**: no hay error, no hay aviso, el vector simplemente representa la primera parte. Por eso comparamos el límite configurado contra el límite de la arquitectura.

**La dimensión** es el tamaño del vector. Afecta el peso del índice y la velocidad de búsqueda, no la calidad por sí sola.

**La arquitectura** confirma que sigue siendo *encoder-only*. Es la evidencia que va en el informe.

**Los parámetros** predicen el tiempo en CPU mejor que cualquier otra cifra.

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer

modelos_cargados = {}
fichas = []

for alias, cfg in MODELOS.items():
    m = SentenceTransformer(cfg["repo"], device="cpu")
    modelos_cargados[alias] = m

    conf = m[0].auto_model.config
    fichas.append({
        "modelo": alias,
        "arquitectura": conf.architectures[0] if conf.architectures else "?",
        "dim": m.get_sentence_embedding_dimension(),
        "max_seq_length": m.max_seq_length,
        "techo_arquitectura": getattr(conf, "max_position_embeddings", None),
        "params_M": round(sum(p.numel() for p in m.parameters()) / 1e6),
        "prefijos": "sí" if cfg["prefijo_consulta"] else "no",
    })

ficha = pd.DataFrame(fichas).set_index("modelo")
ficha

**Cómo leer esta tabla.**

Compara `max_seq_length` con `techo_arquitectura`. Si en granite el primero dice 512 y el segundo dice mucho más, los 32k tokens son teóricos: en la práctica los tres modelos truncan igual y ese criterio deja de diferenciarlos. Si `max_seq_length` sí es grande, granite te permite chunks mucho más largos que e5.

En `arquitectura` esperas algo terminado en `Model` sobre base BERT, ModernBERT o XLM-RoBERTa. Cualquier cosa que mencione `ForCausalLM` sería un decoder y quedaría descalificada.

Nota que `max_seq_length` se mide en **tokens**, no en palabras. En español un token equivale a algo menos de media palabra, así que 512 tokens son más o menos 200 a 300 palabras según el texto.

In [ ]:
# Cuántos tokens gasta realmente cada tokenizer, por idioma.
# Sirve para traducir el límite de tokens a un límite de palabras usable en la Fase 3.
filas_tok = []
for alias, m in modelos_cargados.items():
    for idioma in sorted({t["language"] for t in TEXTOS}):
        muestras = [t["text"] for t in TEXTOS if t["language"] == idioma]
        razones = [
            len(m.tokenizer.encode(x, add_special_tokens=False)) / len(x.split())
            for x in muestras
        ]
        filas_tok.append({
            "modelo": alias,
            "idioma": idioma,
            "tokens_por_palabra_medio": round(sum(razones) / len(razones), 2),
            "tokens_por_palabra_max": round(max(razones), 2),
            "palabras_que_caben": int(m.max_seq_length / max(razones)),
        })

pd.DataFrame(filas_tok).set_index(["modelo", "idioma"])

`palabras_que_caben` es el número que le tienes que pasar a la Fase 3: cuántas palabras entran sin que el modelo trunque, usando el caso más costoso observado. Si sale cómodamente por encima de 250, el límite duro del reto manda y el del encoder no molesta.

## Utilidades de medición

Tres funciones que usan los bloques 2 y 3.

`puntuar` convierte consultas y fragmentos en vectores y devuelve la similitud de todos contra todos. Con `normalize_embeddings=True` el producto punto **es** la similitud coseno, que es justamente lo que hará FAISS con `IndexFlatIP`. Así el notebook mide lo mismo que va a medir el índice real.

`ranking` ordena los fragmentos para una consulta y marca cuáles eran los correctos.

`resumir` saca cuatro números por modelo. Los explico debajo del bloque 2.

In [ ]:
import numpy as np
import pandas as pd

def puntuar(modelo, cfg, consultas, textos):
    '''Matriz (consultas x textos) de similitud coseno.'''
    vq = modelo.encode(
        [cfg["prefijo_consulta"] + c["text"] for c in consultas],
        normalize_embeddings=True, batch_size=8, show_progress_bar=False,
    )
    vt = modelo.encode(
        [cfg["prefijo_texto"] + t["text"] for t in textos],
        normalize_embeddings=True, batch_size=8, show_progress_bar=False,
    )
    return np.asarray(vq) @ np.asarray(vt).T


def ranking(matriz, consultas, textos, id_consulta):
    i = next(k for k, c in enumerate(consultas) if c["id"] == id_consulta)
    filas = [
        {
            "puesto": 0,
            "fragmento": t["id"],
            "idioma": t["language"],
            "relevancia": t["relevance"].get(id_consulta, 0),
            "puntaje": round(float(matriz[i, j]), 4),
        }
        for j, t in enumerate(textos)
    ]
    filas.sort(key=lambda f: -f["puntaje"])
    for puesto, f in enumerate(filas, start=1):
        f["puesto"] = puesto
    return pd.DataFrame(filas).set_index("puesto")


def resumir(matriz, consultas, textos, etiqueta):
    reciprocos, aciertos1, margenes, sesgos = [], [], [], []

    for i, c in enumerate(consultas):
        puntajes = matriz[i]
        rel = np.array([t["relevance"].get(c["id"], 0) for t in textos])
        idiomas = np.array([t["language"] for t in textos])

        orden = np.argsort(-puntajes)
        posiciones_oro = [p for p, j in enumerate(orden, start=1) if rel[j] >= 2]
        reciprocos.append(1 / posiciones_oro[0] if posiciones_oro else 0.0)
        aciertos1.append(1 if rel[orden[0]] >= 2 else 0)

        oro, ruido = puntajes[rel >= 2], puntajes[rel < 2]
        margenes.append(float(oro.min() - ruido.max()) if len(oro) and len(ruido) else np.nan)

        # Sesgo de idioma: solo entre fragmentos NO relevantes, para que la
        # diferencia no se explique por el contenido.
        ruido_es = puntajes[(rel < 2) & (idiomas == "es")]
        ruido_en = puntajes[(rel < 2) & (idiomas == "en")]
        sesgos.append(float(ruido_es.mean() - ruido_en.mean()) if len(ruido_es) and len(ruido_en) else np.nan)

    return {
        "config": etiqueta,
        "MRR": round(float(np.mean(reciprocos)), 3),
        "acierto@1": f"{sum(aciertos1)}/{len(consultas)}",
        "margen": round(float(np.nanmean(margenes)), 4),
        "sesgo_idioma": round(float(np.nanmean(sesgos)), 4),
    }

## Bloque 2 — Idiomas cruzados

Las 50 consultas del reto están en español. Los corpus de los fenómenos 1 y 2 están en inglés. Si un modelo prefiere los documentos que están en el mismo idioma de la consulta, va a subir documentos en español que no responden nada y a enterrar los documentos en inglés que sí responden. Ese sesgo te hunde el F1@3 sin que ningún error lo delate.

Las tres consultas se comparan contra los doce fragmentos a la vez, no solo contra los suyos. Los fragmentos de las otras consultas actúan como distractores, y varios están en otro idioma: es ahí donde el sesgo se vuelve visible.

In [ ]:
matrices = {}
resultados_b2 = []

for alias, m in modelos_cargados.items():
    matrices[alias] = puntuar(m, MODELOS[alias], CONSULTAS, TEXTOS)
    resultados_b2.append(resumir(matrices[alias], CONSULTAS, TEXTOS, alias))

pd.DataFrame(resultados_b2).set_index("config")

**Cómo leer cada columna.**

**`MRR`** — dónde queda el primer fragmento correcto. Vale 1.0 si sale de primero, 0.5 si de segundo, 0.33 si de tercero. Con 12 candidatos, cualquier cosa por debajo de 0.5 es mala señal.

**`acierto@1`** — en cuántas de las tres consultas el primer resultado era correcto. Es crudo pero es lo que más se parece a la experiencia real.

**`margen`** — la distancia entre el peor fragmento correcto y el mejor incorrecto. **Positivo** significa que hay una separación limpia: todos los correctos por encima de todos los demás. **Negativo** significa que se mezclan. Este número importa más que el MRR para elegir el umbral θ de similitud en la Fase 6: si el margen es amplio, hay un corte que separa bien; si es negativo, ningún umbral te salva.

**`sesgo_idioma`** — la diferencia de puntaje entre distractores en español y distractores en inglés. Solo mira fragmentos **no relevantes**, así que el contenido no explica la diferencia: si sale un número grande, es el idioma. Cerca de 0 es lo ideal. **Positivo** quiere decir que el modelo infla los textos en español solo por estar en español, que es el escenario que te perjudica.

Ahora el detalle consulta por consulta. Es donde se ven las cosas que un promedio esconde.

In [ ]:
for id_consulta in [c["id"] for c in CONSULTAS]:
    texto = next(c["text"] for c in CONSULTAS if c["id"] == id_consulta)
    print("=" * 100)
    print(f"{id_consulta}: {texto[:95]}")
    print("=" * 100)
    for alias in modelos_cargados:
        print(f"\n-- {alias} --")
        print(ranking(matrices[alias], CONSULTAS, TEXTOS, id_consulta).head(5).to_string())
    print()

En estas tablas busca tres cosas:

1. ¿Los fragmentos con relevancia 3 están en los primeros puestos?
2. ¿Hay fragmentos de **otra** consulta metidos arriba? Eso es confusión temática entre fenómenos.
3. ¿Los puntajes están muy juntos? Si el primero saca 0.87 y el quinto 0.85, el modelo no está distinguiendo nada; el orden es casi ruido.

## Bloque 3 — Prefijos de e5

`multilingual-e5-base` se entrenó con dos etiquetas pegadas al texto: `query:` delante de la consulta y `passage:` delante del documento. No son adorno, forman parte de la entrada que el modelo espera.

Esto se mide en vez de citarse, por dos razones. Una: si la caída es grande, olvidar los prefijos en el `generador.py` sería un error catastrófico y silencioso. Dos: el número entra en el informe técnico como justificación.

Se prueban tres configuraciones sobre el mismo modelo: con prefijos, sin ninguno, y con los prefijos intercambiados a propósito.

In [ ]:
variantes = {
    "e5 con prefijos":      {"prefijo_consulta": "query: ",   "prefijo_texto": "passage: "},
    "e5 sin prefijos":      {"prefijo_consulta": "",          "prefijo_texto": ""},
    "e5 prefijos al revés": {"prefijo_consulta": "passage: ", "prefijo_texto": "query: "},
}

resultados_b3 = [
    resumir(puntuar(modelos_cargados["e5-base"], cfg, CONSULTAS, TEXTOS), CONSULTAS, TEXTOS, nombre)
    for nombre, cfg in variantes.items()
]

pd.DataFrame(resultados_b3).set_index("config")

Si las tres filas salen casi idénticas, no es que los prefijos den igual: es que con tres consultas y doce fragmentos la prueba no tiene resolución suficiente. En ese caso, deja los prefijos puestos igual —son los que recomienda el autor del modelo— y anota en el informe que el efecto no fue medible a esta escala.

## Bloque 4 — Tiempo en CPU

Sin GPU, este número puede pesar más que cualquier diferencia de calidad. No es solo la corrida final: vas a reindexar cada vez que cambies el chunking, arregles un extractor o ajustes un parámetro. Un modelo que tarda seis horas por pasada te deja una prueba por día.

Se mide el costo por fragmento y se extrapola. Ajusta `CHUNKS_ESTIMADOS` cuando sepas el tamaño real del corpus: si te salen 400 documentos con un promedio de 60 chunks cada uno, son 24.000.

In [ ]:
import time

CHUNKS_ESTIMADOS = 30_000
REPETICIONES = 3

filas_tiempo = []
for alias, m in modelos_cargados.items():
    cfg = MODELOS[alias]
    entradas = [cfg["prefijo_texto"] + t["text"] for t in TEXTOS]

    m.encode(entradas[:2], normalize_embeddings=True, show_progress_bar=False)  # calentar

    inicio = time.perf_counter()
    for _ in range(REPETICIONES):
        m.encode(entradas, normalize_embeddings=True, batch_size=8, show_progress_bar=False)
    transcurrido = time.perf_counter() - inicio

    por_fragmento = transcurrido / (REPETICIONES * len(entradas))
    filas_tiempo.append({
        "modelo": alias,
        "ms_por_fragmento": round(por_fragmento * 1000, 1),
        "fragmentos_por_seg": round(1 / por_fragmento, 1),
        "horas_corpus_completo": round(por_fragmento * CHUNKS_ESTIMADOS / 3600, 2),
    })

pd.DataFrame(filas_tiempo).set_index("modelo")

Dos advertencias sobre esta extrapolación.

Los fragmentos del mini-corpus rondan las 200 palabras. Si en la Fase 3 terminas con chunks de 140, el costo real baja. La estimación es conservadora.

Y el costo no es lineal con el número de parámetros: depende también de cuántos núcleos tenga tu portátil y de si hay otra cosa corriendo. Corre esta celda con el equipo tranquilo.

## Resumen para decidir

In [ ]:
resumen = (
    pd.DataFrame(resultados_b2).set_index("config")
      .join(pd.DataFrame(filas_tiempo).set_index("modelo")[["ms_por_fragmento", "horas_corpus_completo"]])
      .join(ficha[["dim", "max_seq_length", "params_M"]])
)
resumen

### Cómo decidir con esta tabla

El orden en que pesan los criterios:

**Primero, descartar por sesgo de idioma.** Un modelo con `sesgo_idioma` alto es inservible acá, por bueno que se vea en todo lo demás: dos de los tres fenómenos están en inglés y todas las consultas en español.

**Segundo, mirar margen antes que MRR.** El MRR con tres consultas es frágil; el margen te dice si los puntajes separan de verdad, y es lo que te va a permitir fijar el umbral θ en la Fase 6.

**Tercero, el tiempo como filtro práctico.** Si un modelo se pasa de unas dos horas por corrida completa en tu máquina, considera si la ganancia en calidad compensa perder la capacidad de iterar.

**Cuarto, la dimensión casi al final.** 384 contra 768 significa la mitad de índice y búsquedas más rápidas. Solo importa si granite-97m queda cerca de los otros en calidad; si empata, gana por barato.

### Antes de cerrar la decisión

Lo que mide este notebook es **tres consultas y doce fragmentos**. Alcanza para descartar un modelo malo, no para elegir con confianza entre dos que quedaron parejos. Si dos modelos empatan:

- El reto permite usar varios encoders y fusionar los resultados con RRF o CombSUM. Dos modelos que se equivocan en cosas distintas fusionados suelen superar al mejor de los dos por separado. Un empate acá es un argumento a favor de la fusión, no un problema.
- La decisión firme se toma con el harness de la Fase 7, sobre las 30 consultas propias con relevancia graduada. Este notebook deja la lista corta.

Anota en el informe el criterio con el que descartaste, no solo el modelo que ganó. Eso es lo que se evalúa.